Carregando os dados e separando df's de treino e de teste.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "dengue_pipeline").is_dir())
sys.path.insert(0, str(PROJECT_ROOT))

from dengue_pipeline import DengueDataCleaner

from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

cleaner = DengueDataCleaner()
df = cleaner.transformar_ml()

df_train = df[(df["notification_year"].isin([2017, 2018])) | ((df["notification_year"] == 2019) & (df["notification_month"] <= 5))].copy()
df_test = df[(df["notification_year"] == 2019) & (df["notification_month"] >= 6)].copy()

print("Treino:", df_train.shape)
print("Teste:", df_test.shape)

print("Proporção treino:", (len(df_train) / len(df))*100)
print("Proporção teste:", (len(df_test) / len(df))*100)

df.columns

Definindo o target e eliminando colunas que não são apropriadas para o modelo

In [ ]:
target = "final_classification"

y_train = df_train[target]
y_test = df_test[target]

cols_remove = ['final_classification']

X_train = df_train.drop(columns = cols_remove)
X_test = df_test.drop(columns=cols_remove)

X_train = X_train.select_dtypes(include=["number"])
X_test = X_test[X_train.columns]

Implementação de Regressão Logistica 

In [ ]:
#Regressao Logistica - testando...
from sklearn.linear_model import LogisticRegression

modelo = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

modelo.fit(X_train, y_train)

y_pred = modelo.predict(X_test)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

Implementação de XGBoost

In [ ]:
#XGBoost testando...

from xgboost import XGBClassifier

modelo = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=6,
        random_state=42,
        eval_metric="logloss",
        n_jobs=-1
    ))
])

modelo.fit(X_train, y_train)

y_pred = modelo.predict(X_test)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

Implementação de Arvore de Decisao

In [ ]:
from sklearn.tree import DecisionTreeClassifier

modelo = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", DecisionTreeClassifier(
        max_depth=10,
        class_weight="balanced",
        random_state=42
    ))
])

modelo.fit(X_train, y_train)

y_pred = modelo.predict(X_test)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))